<a href="https://colab.research.google.com/github/liqq1024/GenAI_LLMs_2026Fall/blob/main/assignments/Homeweek1%20/Homework1_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 1 Solution: Tokenization, Embeddings, and Sequence Modeling with PyTorch

This instructor solution follows the published Homework 1 requirements. Run the notebook from top to bottom. Because the dataset is intentionally small, exact validation metrics and generated text may vary slightly across environments.

## Setup

Google Colab already includes PyTorch. The next cell installs the remaining libraries used for tokenization and PCA.

In [ ]:
!pip -q install transformers tokenizers datasets scikit-learn

import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from transformers import AutoTokenizer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Part 1: Tokenization with GPT-2 and BERT

In [ ]:
text = "Large Language Models are changing AI."

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

gpt2_tokens = gpt2_tokenizer.tokenize(text)
gpt2_ids = gpt2_tokenizer.encode(text, add_special_tokens=False)
bert_tokens = bert_tokenizer.tokenize(text)
bert_ids = bert_tokenizer.encode(text, add_special_tokens=False)

print("GPT-2 tokens:", gpt2_tokens)
print("GPT-2 token IDs:", gpt2_ids)
print("BERT tokens:", bert_tokens)
print("BERT token IDs:", bert_ids)

### Tokenizer comparison and reference answers

| Tokenizer | Tokens | Token IDs | Number of tokens |
|---|---|---|---:|
| GPT-2 | `['Large', 'ĠLanguage', 'ĠModels', 'Ġare', 'Ġchanging', 'ĠAI', '.']` | Produced in the cell above | 7 |
| BERT | `['large', 'language', 'models', 'are', 'changing', 'ai', '.']` | Produced in the cell above | 7 |

1. The tokenizers do not produce exactly the same token strings. GPT-2 uses `Ġ` to indicate a preceding space (for example, `ĠLanguage`), while BERT lowercases this example (for example, `language`).
2. A neural network operates on numerical tensors, so token IDs provide an indexable numerical representation of text.
3. For a GPT-style text-generation application, use the GPT-2 tokenizer because its vocabulary and tokenization scheme match the model family.
4. Tokenization determines sequence length and how text fragments are represented. It can affect computational cost, vocabulary coverage, and how easily a model represents uncommon words.

## Part 2: Create a Dataset for Next-Word Prediction

The custom word-level vocabulary below is separate from the pre-trained GPT-2 and BERT vocabularies used in Part 1.

In [ ]:
sentences = [
    "healthy plants need sunlight and water",
    "disease symptoms can appear on leaves",
    "early detection helps protect crops",
    "machine learning can identify plant diseases",
    "farmers use images to monitor plant health",
    "deep learning models learn patterns from data",
    "sensors can support precision agriculture",
    "timely treatment can reduce crop damage",
    "leaf spots may indicate fungal infection",
    "data quality affects model performance",
]

# 1--3. Tokenize at word level and create a custom vocabulary.
tokenized_sentences = [sentence.lower().split() for sentence in sentences]
special_tokens = ["<PAD>", "<UNK>"]
vocab_words = sorted({word for sentence in tokenized_sentences for word in sentence})
id_to_word = special_tokens + vocab_words
word_to_id = {word: idx for idx, word in enumerate(id_to_word)}

PAD_ID = word_to_id["<PAD>"]
UNK_ID = word_to_id["<UNK>"]
vocab_size = len(word_to_id)

encoded_sentences = [
    [word_to_id.get(word, UNK_ID) for word in sentence]
    for sentence in tokenized_sentences
]

# 4. Each prefix predicts the next word.
input_sequences, targets = [], []
for sentence_ids in encoded_sentences:
    for position in range(1, len(sentence_ids)):
        input_sequences.append(torch.tensor(sentence_ids[:position], dtype=torch.long))
        targets.append(sentence_ids[position])

# 5. Right-padding lets every sequence have the same length.
X = pad_sequence(input_sequences, batch_first=True, padding_value=PAD_ID)
y = torch.tensor(targets, dtype=torch.long)

# 6. Reproducible 80/20 train-validation split.
indices = list(range(len(X)))
random.shuffle(indices)
split_index = int(0.8 * len(indices))
train_indices, val_indices = indices[:split_index], indices[split_index:]
X_train, y_train = X[train_indices], y[train_indices]
X_val, y_val = X[val_indices], y[val_indices]

# 7. PyTorch datasets and loaders.
batch_size = 8
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

example_index = 3
print("Vocabulary size:", vocab_size)
print("Original sentence:", sentences[example_index])
print("Word-level token IDs:", encoded_sentences[example_index])
print("Example input IDs:", input_sequences[0].tolist())
print("Example input words:", [id_to_word[i] for i in input_sequences[0].tolist()])
print("Target word:", id_to_word[targets[0]])
print("X_train:", tuple(X_train.shape), " y_train:", tuple(y_train.shape))
print("X_val:  ", tuple(X_val.shape), " y_val:  ", tuple(y_val.shape))

### Part 2 reference answers

1. Text must be converted to numbers because neural networks perform mathematical operations on numerical tensors.
2. A word ID is a unique integer index for a word in the custom vocabulary. It is a label, not a quantity with intrinsic meaning.
3. Padding gives all sequences in a batch a common length so they can be processed together.
4. The target is the ID of the next word, given the preceding word IDs as input.

## Shared training and evaluation helpers

In [ ]:
def evaluate_model(model, data_loader, criterion):
    model.eval()
    total_loss = total_correct = total_examples = 0
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_examples += y_batch.size(0)
    return total_loss / total_examples, total_correct / total_examples


def train_model(model, train_loader, val_loader, epochs=100, learning_rate=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        total_loss = total_correct = total_examples = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_examples += y_batch.size(0)

        train_loss = total_loss / total_examples
        train_acc = total_correct / total_examples
        val_loss, val_acc = evaluate_model(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        if epoch == 0 or (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch + 1:3d}/{epochs} | train loss {train_loss:.3f}, acc {train_acc:.1%} | val loss {val_loss:.3f}, acc {val_acc:.1%}")
    return history


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="Training")
    axes[0].plot(epochs, history["val_loss"], label="Validation")
    axes[0].set(title=f"{title}: Loss", xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[1].plot(epochs, history["train_acc"], label="Training")
    axes[1].plot(epochs, history["val_acc"], label="Validation")
    axes[1].set(title=f"{title}: Accuracy", xlabel="Epoch", ylabel="Accuracy")
    for ax in axes:
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Part 3: Train a SimpleRNN Model

In [ ]:
class SimpleRNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_id):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lengths = (x != self.pad_id).sum(dim=1).cpu()
        packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
        _, hidden = self.rnn(packed)
        return self.output_layer(hidden[-1])


embedding_dim, hidden_dim, num_epochs = 32, 64, 100
model_rnn = SimpleRNNModel(vocab_size, embedding_dim, hidden_dim, PAD_ID).to(device)
print(model_rnn)
rnn_history = train_model(model_rnn, train_loader, val_loader, epochs=num_epochs)
plot_history(rnn_history, "SimpleRNN")
print(f"Final SimpleRNN training accuracy: {rnn_history['train_acc'][-1]:.2%}")
print(f"Final SimpleRNN validation accuracy: {rnn_history['val_acc'][-1]:.2%}")

### Part 3 reference answers

1. `nn.Embedding` maps each word ID to a learned dense vector. During training, vectors can adapt to represent useful patterns of word usage.
2. The RNN hidden state summarizes earlier words in the sequence and uses that summary when predicting the next word.
3. Interpret the plotted curves. On this tiny corpus, training accuracy can become high while validation accuracy fluctuates; a persistent gap is evidence of overfitting. Exact results depend on the split and random initialization.

## Part 4: Train an LSTM Model

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_id):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lengths = (x != self.pad_id).sum(dim=1).cpu()
        packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
        _, (hidden, cell) = self.lstm(packed)
        return self.output_layer(hidden[-1])


model_lstm = LSTMModel(vocab_size, embedding_dim, hidden_dim, PAD_ID).to(device)
print(model_lstm)
lstm_history = train_model(model_lstm, train_loader, val_loader, epochs=num_epochs)
plot_history(lstm_history, "LSTM")
print(f"Final LSTM training accuracy: {lstm_history['train_acc'][-1]:.2%}")
print(f"Final LSTM validation accuracy: {lstm_history['val_acc'][-1]:.2%}")

## Part 5: Visualize and Analyze Embeddings

In [ ]:
if rnn_history["val_acc"][-1] >= lstm_history["val_acc"][-1]:
    best_model, best_model_name = model_rnn, "SimpleRNN"
else:
    best_model, best_model_name = model_lstm, "LSTM"

embedding_matrix = best_model.embedding.weight.detach().cpu().numpy()
selected_words = ["healthy", "plants", "disease", "symptoms", "learning", "models", "data", "sensors", "crops", "health"]
selected_words = [word for word in selected_words if word in word_to_id]
selected_ids = [word_to_id[word] for word in selected_words]
selected_embeddings = embedding_matrix[selected_ids]

pca = PCA(n_components=2)
embedding_2d = pca.fit_transform(selected_embeddings)
plt.figure(figsize=(9, 6))
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], s=100, alpha=0.75)
for i, word in enumerate(selected_words):
    plt.annotate(word, embedding_2d[i], xytext=(5, 5), textcoords="offset points")
plt.title(f"Word Embeddings from the {best_model_name} Model (PCA)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(alpha=0.3)
plt.show()
print("PCA explained variance ratio:", pca.explained_variance_ratio_)

query_word = "disease"
query_embedding = embedding_matrix[word_to_id[query_word]].reshape(1, -1)
similarity_scores = cosine_similarity(query_embedding, embedding_matrix)[0]
neighbors = []
for word_id in np.argsort(similarity_scores)[::-1]:
    candidate = id_to_word[word_id]
    if candidate not in {query_word, "<PAD>", "<UNK>"}:
        neighbors.append((candidate, similarity_scores[word_id]))
    if len(neighbors) == 3:
        break
print(f"Three nearest neighbors of '{query_word}':")
for word, score in neighbors:
    print(f"  {word}: {score:.4f}")

### Part 5 reference answers

1. Identify close words from the displayed PCA plot; the actual locations depend on the trained embedding weights.
2. The nearest-neighbor list may make partial semantic sense when words share contexts in the corpus, but relationships are unreliable with only ten sentences.
3. Word embeddings are learned vectors that can encode patterns of context and usage. In contrast, adjacent word IDs are not inherently similar.
4. PCA reduces a higher-dimensional representation to two dimensions, so it necessarily loses information and can distort distances.

## Part 6: Compare the Models

In [ ]:
print("| Criterion | SimpleRNN | LSTM |")
print("|---|---:|---:|")
print(f"| Final training accuracy | {rnn_history['train_acc'][-1]:.2%} | {lstm_history['train_acc'][-1]:.2%} |")
print(f"| Final validation accuracy | {rnn_history['val_acc'][-1]:.2%} | {lstm_history['val_acc'][-1]:.2%} |")

### Part 6 reference answers

1. Use the printed final validation accuracies to identify the stronger model in this run.
2. Compare the text examples produced in Part 7; the more coherent result is the one that better follows the seed phrase and corpus patterns.
3. An LSTM has a cell state plus input, forget, and output gates. These mechanisms help it retain or discard information over longer sequences, whereas a simple RNN can lose earlier information during repeated updates.
4. Use the loss and accuracy curves as evidence. A high training score with substantially lower or worsening validation performance indicates overfitting. The small validation set makes these values unstable.
5. In both parts, IDs are lookup indexes. GPT-2/BERT IDs index pre-trained vocabularies, while the custom IDs index the homework vocabulary. The embedding layer turns each ID into a learned vector.

## Part 7: Generate Text

In [ ]:
def generate_text(model, seed_phrase, num_new_words=5):
    model.eval()
    generated_ids = [word_to_id.get(word.lower(), UNK_ID) for word in seed_phrase.split()]
    max_sequence_length = X.shape[1]
    with torch.no_grad():
        for _ in range(num_new_words):
            input_ids = generated_ids[-max_sequence_length:]
            padded_ids = input_ids + [PAD_ID] * (max_sequence_length - len(input_ids))
            logits = model(torch.tensor([padded_ids], dtype=torch.long, device=device))
            next_id = logits.argmax(dim=1).item()
            if next_id in {PAD_ID, UNK_ID}:
                break
            generated_ids.append(next_id)
    return " ".join(id_to_word[i] for i in generated_ids)


seed_phrases = ["machine learning", "disease symptoms"]
for seed in seed_phrases:
    print(f"\nSeed phrase: {seed}")
    print("SimpleRNN:", generate_text(model_rnn, seed))
    print("LSTM:     ", generate_text(model_lstm, seed))

### Part 7 reference discussion

Use the generated outputs above in the comparison table. With this small corpus, either model may repeat words or generate an incomplete phrase. The more meaningful model is the one whose continuation better reflects the seed phrase and patterns present in the training sentences.

## Part 8: Reference Reflection

This homework shows the full path from text to sequential prediction. GPT-2 and BERT use different tokenization conventions, but both convert text into token IDs that models can process. In the PyTorch portion, the custom vocabulary provides a word ID for each word in the small corpus. These IDs are only indexes; the embedding layer learns dense vectors that can encode useful relationships between words based on training context.

The SimpleRNN and LSTM both process sequences one word at a time. The SimpleRNN carries information in a single hidden state, while the LSTM uses a cell state and gates to control what information should be retained, updated, or forgotten. This design can make LSTMs more effective when an earlier word remains relevant later in a sequence. In this experiment, however, the corpus is extremely small, so validation metrics and generated text are not strong evidence of general language ability. The models can memorize short patterns, and the validation split contains few examples. With a larger, more diverse corpus, I would use a fixed train/validation/test split, tune hyperparameters, and evaluate generated text more systematically. I would also consider subword tokenization and more modern transformer-based models for stronger language modeling.